# Analyzer Agent Evaluation Notebook
This notebook demonstrates how to evaluate the `ANALYZER` agent from `creator_agent.py` using LangSmith.

**Note on File Paths:** For simplicity, this notebook creates mock artifact files in the locations the agent expects by default (i.e., within the `agents/agentEvolver_v2/runs/...` directory), rather than inside this `evaluation` folder. This avoids the need to refactor the agent's file-reading tools.

In [8]:
import os
import json
from pathlib import Path
import sys
import uuid
from langsmith import Client
from langsmith.evaluation import evaluate
from langchain_core.messages import HumanMessage, AIMessage
from typing import TypedDict, List, Any
from pydantic import BaseModel, Field

# Adjust path to import from the parent directory
module_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from creator_agent import (CreatorAgent, CreatorGraphState, read_foo, read_full_performance_history, read_game_output_file, read_game_results_file, DEFAULT_ANALYZE_MSG)
from prompts import ANALYZER_SYSTEM_PROMPT

print("Please ensure your LANGSMITH_API_KEY and LANGCHAIN_TRACING_V2 environment variables are set.")
os.environ['LANGCHAIN_TRACING_V2'] = 'true'

# Instantiate the CreatorAgent
creator_agent = CreatorAgent()
run_dir = Path(creator_agent.run_dir)
print(f"CreatorAgent run directory: {run_dir}")

Please ensure your LANGSMITH_API_KEY and LANGCHAIN_TRACING_V2 environment variables are set.
CreatorAgent run directory: /Users/dakotabarnes/Develop/CollectiveComputingLabs/strategy-game-agents/agents/agentEvolver_v2/runs/creator_20250909_001018


In [ ]:
# --- Create Mock Data Files ---
# The analyzer's tools read files from specific locations. We create them here.

# Create a dummy game run directory inside the main run_dir
game_run_id = "game_eval_run_fg"
game_run_dir = run_dir / game_run_id
game_run_dir.mkdir(exist_ok=True)

# 1. Dummy game_output.txt for the history
game_output_content = """GAME RESULTS:\nPlayer FooPlayer had an error.\nDefaulting to Random Action.\nChoose action with score: 0"""
with open(game_run_dir / "game_output.txt", "w") as f:
    f.write(game_output_content)

# 2. Dummy performance_history.json
performance_history_content = {
    "Evolution 0": {
        "wins": 0,
        "avg_score": 0,
        "avg_turns": 0,
        "full_game_log_path": str((game_run_dir / "game_output.txt").relative_to(run_dir)),
        "json_game_results_path": "None",
        "cur_foo_player_path": "None",
        "timestamp": "2025-09-08 23:20:00"
    }
}
with open(run_dir / "performance_history.json", "w") as f:
    json.dump(performance_history_content, f, indent=2)



print("Mock data files created.")

Mock data files created.


In [10]:
# --- LangSmith Dataset Setup ---
client = Client()
dataset_name = "Analyzer_Agent_Evaluation_v2"

if client.has_dataset(dataset_name=dataset_name):
    client.delete_dataset(dataset_name=dataset_name)

dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Evaluation for the Analyzer agent's ability to parse game logs.",
)

# The input for our target function is the initial state of the graph
mock_input_state = {
    "analyzer_messages": [],
    "recent_meta_message": HumanMessage(content=DEFAULT_ANALYZE_MSG.format(FOO_TARGET_FILENAME='foo_player.py')),
    "meta_messages": [], "strategizer_messages": [], "researcher_messages": [], "coder_messages": [],
    "recent_helper_response": AIMessage(content=''), "game_results": HumanMessage(content=''), "tool_calling_messages": []
}

# The reference output for our evaluators
reference_output = {"criteria": ["Defaulting to Random Action", "Choose action with score: 0"]}

client.create_example(
    inputs=mock_input_state,
    outputs=reference_output,
    dataset_id=dataset.id,
)

print(f"Dataset '{dataset_name}' created and example added.")

Dataset 'Analyzer_Agent_Evaluation_v2' created and example added.


In [11]:
# --- Evaluator Definitions ---

class Criteria(BaseModel):
    criteria_text: str = Field(description="The specific success criteria being evaluated.")
    reasoning: str = Field(description="Detailed explanation of why this criteria is or isn't captured.")
    is_captured: bool = Field(description="Whether this specific criteria is adequately captured.")

BRIEF_CRITERIA_PROMPT = ANALYZER_SYSTEM_PROMPT

def evaluate_success_criteria(run, example):
    # Simplified version for this context
    prediction = run.outputs['recent_helper_response'].content
    success_criteria = example.outputs["criteria"]
    
    captured_count = 0
    for criterion in success_criteria:
        if criterion in prediction:
            captured_count += 1
    
    score = captured_count / len(success_criteria) if success_criteria else 0.0
    return {"key": "success_criteria_score", "score": score}

# --- Target Function and Experiment Run ---

def target_func(inputs):
    # The analyzer_node returns the full updated state
    return creator_agent.analyzer_node(inputs)

experiment_results = evaluate(
    target_func,
    data=dataset_name,
    evaluators=[evaluate_success_criteria],
    experiment_prefix="Analyzer Agent Test",
    description="Testing the analyzer's ability to detect errors in logs."
)

# Print results locally
print("\n--- Local Evaluation Results ---")
print(experiment_results.to_pandas())

View the evaluation results for experiment: 'Analyzer Agent Test-6ba04235' at:
https://smith.langchain.com/o/174296c9-66e9-46df-b863-32bcff3db16f/datasets/88f8d3ca-2d41-4154-bd7e-5515e2cb9c8a/compare?selectedSessions=6c299298-225d-4cc6-be04-e6212b606efd




0it [00:00, ?it/s]

================================ Human Message =================================


ANALYZER OBJECTIVE:

If there is no syntax errors, I want you to return
- The Scores of the foo_player.py player from the game_results json file
- Short analysis of the game output (return anything interseting that was printed)
- EMPHASIZE any errors, warnings, or signs of player implementation error in the game_output.txt file

If there is a syntax error, I want you to return
- The error message from the game_output.txt file
- The exact line number of the error if possible
- The exact line of code that caused the error if possible

Keep the Response as concise as possible
Start your response with "After Running The New foo_player.py Player, Here is my analysis and findings:"



================================== Ai Message ==================================
Tool Calls:
  read_local_file (cdb4w3Zym)
 Call ID: cdb4w3Zym
  Args:
    rel_path: game_eval_run_fg/game_output.txt


=============================

1it [00:07,  7.20s/it]

================================== Ai Message ==================================

ANALYZER
After Running The New foo_player.py Player, Here is my analysis and findings:
---
**Scores from game_results json file**: No scores available (No game-results file recorded for Evolution 0).

**Game Output Analysis**:
- **Error**: `Player FooPlayer had an error.`
- **Warning**: `Defaulting to Random Action.` (Indicates the player's `choose_action` method returned `None`, forcing the game to use a random action.)
- **Debugging Info**: `Choose action with score: 0` (Suggests the player's action selection logic is either missing or ineffective.)

**Root Cause**: The `choose_action` method in `foo_player.py` returns `None`, which is invalid and triggers the game to default to a random action.

Let me know if you need anything else
END ANALYZER




1it [00:07,  7.51s/it]


--- Local Evaluation Results ---
                                 inputs.game_results inputs.meta_messages  \
0  {'type': 'human', 'content': '', 'example': Fa...                   []   

  inputs.coder_messages inputs.analyzer_messages  \
0                    []                       []   

                          inputs.recent_meta_message  \
0  {'type': 'human', 'content': '
ANALYZER OBJECT...   

  inputs.researcher_messages inputs.strategizer_messages  \
0                         []                          []   

  inputs.tool_calling_messages  \
0                           []   

                       inputs.recent_helper_response  \
0  {'type': 'ai', 'content': '', 'example': False...   

                      outputs.recent_helper_response  \
0  content="ANALYZER\nAfter Running The New foo_p...   

                       outputs.tool_calling_messages  \
0  [content='This is the current performance hist...   

                               outputs.meta_messages  \
0  [cont